# D2. 표본 흔들림 — 다른 6만 문장을 뽑아도 비슷한 점수가 나오는가

**질문**: split_v1의 6만은 무작위로 뽑은 하나의 표본이다. 우연히 뽑힌 문장 때문에 점수가 높거나 낮게 나온 건 아닌가?

**설계**
- 서로 **한 문장도 겹치지 않는** 6만 세트 A(=split_v1 train), B, C로 같은 모델을 학습한다.
- dev·test, 모델, 설정, 시드는 모두 고정. **학습 세트만** 바꾼다.
- 추가로 세트 A에서 시드만 바꿔(42, 1, 2) 학습 과정 자체의 흔들림도 잰다.

| 확인하는 흔들림 | 바꾸는 것 | 고정하는 것 |
|---|---|---|
| 표본 선택의 우연 | 학습 세트 A·B·C | 시드, dev, test |
| 학습 과정의 우연 | 시드 42·1·2 | 데이터 |

**쓰는 곳**: M1에서 두 모델의 test F1 차이가 이 흔들림보다 작으면 "차이가 있다"고 말하지 않는다. 흔들림이 크게 나오면 상위 모델 두 개를 B·C에서도 돌려 순위가 유지되는지 본다.

## 0. 공통 준비 (모든 노트북 같음)
Colab → 런타임 → 런타임 유형 변경 → GPU(T4 이상). 구글 드라이브를 연결하면 이 PC의 프로젝트 폴더가 `/content/drive/Othercomputers/내 PC/...` 아래에 보인다. 경로가 다르면 왼쪽 파일 탭에서 찾아 `PROJECT` 만 고친다.

공용 코드 `src/ner_lib.py`(학습)와 `src/evaluate.py`(팀 공용 채점기)를 불러온다. 정렬·복원·채점 규칙이 모든 노트북에서 같아야 비교가 공정하기 때문이다.

**저장되는 것**: 실험마다 `reports/results/{실험}.json`(에폭별 dev 점수 포함), `preds/`(예측), `scores/`(채점), `errors/`(오답), 그리고 `runs.jsonl`(실행 기록 한 줄). 그림은 R0 노트북이 이 파일들로 그린다.
**끊겼을 때**: 같은 셀을 다시 실행하면 끝난 실험은 건너뛰고, 진행 중이던 실험은 마지막 에폭 체크포인트부터 이어서 학습한다.

**최종 가중치(10/23부터 항상 저장)**: 학습이 끝나면 예측·채점보다 먼저 `내 드라이브/ner_models/{실험}/` 에 가중치(model.safetensors 또는 crf_model_state.pt)와 토크나이저를 저장하고, 다시 읽어 학습된 값과 같은지 확인한다. 확인에 실패하면 체크포인트를 지우지 않고 멈춘다. 런타임 반납 전에도 한 번 더 확인한다.

In [ ]:
!pip -q install -U transformers datasets accelerate pytorch-crf kiwipiepy
import sys, os, json, torch
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/Othercomputers/내 PC/NLP/프로젝트1/공통2_개체명인식_NER"
sys.path.insert(0, f"{PROJECT}/src")
import importlib, ner_lib; importlib.reload(ner_lib)
MODELS = "/content/drive/MyDrive/ner_models"   # 최종 가중치(실험마다 폴더). 학습 직후 저장하고 다시 읽어 확인한다. 지우지 말 것
ner_lib.DEFAULTS.update(model_dir=MODELS)
from ner_lib import run_experiment, load_results, results_table, upper_bound, load_jsonl, prepare_aihub_model
SPLIT   = f"{PROJECT}/data/split_v1"          # dev 1만 / test 1만 (모든 버전 공통)
TRAIN   = f"{PROJECT}/data/split_v3/train.jsonl"   # 팀 기준 학습 데이터: 30만 (2026-10-22 결정)
SUFFIX  = "_300k"                              # 30만으로 학습한 실험 이름 꼬리표
RESULTS = f"{PROJECT}/reports/results"        # 결과 json이 모이는 곳 (팀원 모두 같은 폴더)
CKPT    = "/content/drive/MyDrive/ner_ckpt"    # 에폭 체크포인트. 끊겨도 이어서 학습. '내 드라이브'라 PC로 동기화되지 않음
MEMBER  = "sejin_choi"                          # ← 실행한 사람 이름, 영어로 (결과 기록 runs.jsonl에 남음)
ner_lib.DEFAULTS.update(ckpt_dir=CKPT, member=MEMBER)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

In [ ]:
ner_lib.DEFAULTS["stage"] = "D2_표본흔들림"
EXP = f"{PROJECT}/data/exp_sets"
BASE = "klue/roberta-base"
COMMON = dict(dev=f"{SPLIT}/dev.jsonl", test=f"{SPLIT}/test.jsonl", out_dir=RESULTS, model=BASE,
              max_len=512, lr=5e-5, epochs=3, batch=16)
# 세트 A·시드 42(= 기준선 M1_klue_roberta)는 GPU 서버가 돌린다 → Colab에서는 아래 4개만
RUNS = {
    "D2_set_B_seed42": dict(train=f"{EXP}/var_train_B.jsonl", seed=42),
    "D2_set_C_seed42": dict(train=f"{EXP}/var_train_C.jsonl", seed=42),
    "D2_set_A_seed1":  dict(train=f"{SPLIT}/train.jsonl", seed=1),
    "D2_set_A_seed2":  dict(train=f"{SPLIT}/train.jsonl", seed=2),
}

## 1. 시작 전 확인
학습에 쓸 파일이 드라이브에 다 있는지 본다. 없으면 PC의 구글 드라이브 동기화가 끝나지 않은 것이므로 잠시 기다렸다가 다시 실행한다. 이미 끝난 실험이 있으면 표시된다(다시 돌리면 건너뜀).

In [ ]:
need = [COMMON["dev"], COMMON["test"]] + [v["train"] for v in RUNS.values()]
for f in sorted(set(need)):
    ok = os.path.exists(f)
    print("있음" if ok else "없음!!", f.replace(PROJECT, "..."), f"{os.path.getsize(f)/1e6:.1f}MB" if ok else "")
assert all(os.path.exists(f) for f in need), "데이터 파일이 아직 드라이브에 없습니다"
for exp in RUNS:
    print(exp, "끝남(건너뜀)" if os.path.exists(f"{RESULTS}/{exp}.json") else "할 차례")

## 2. 실행 (T4 기준 실험당 30~50분, 4개 2~3시간)
연결이 끊기면 0번 셀부터 다시 실행한다. 끝난 실험은 건너뛰고, 진행 중이던 실험은 '내 드라이브/ner_ckpt' 의 마지막 에폭 체크포인트부터 이어서 학습한다.

In [ ]:
for exp, v in RUNS.items():
    run_experiment({**COMMON, **v, "exp": exp})

## 3. 결과 해석
표준편차(std)는 점수가 평균에서 보통 얼마나 벗어나는지를 뜻한다. 대략 **두 모델 차이가 std의 2배보다 작으면** 우연으로 생길 수 있는 차이로 본다. 세트 A·시드 42 결과는 서버에서 가져와 합친 뒤에 나오므로, 지금은 B·C와 시드 1·2만 보일 수 있다.

In [ ]:
import statistics as st
rs = {r["exp"]: r for r in load_results(RESULTS) if r["exp"].startswith("D2_")}
def summary(names, label):
    xs = [rs[n]["test"]["ALL"]["f1"] for n in names if n in rs]
    if len(xs) >= 2:
        print(f"{label}: {[round(x,4) for x in xs]} → 평균 {st.mean(xs):.4f}, std {st.stdev(xs):.4f}, 범위 {max(xs)-min(xs):.4f}")
summary(["D2_set_A_seed42", "D2_set_B_seed42", "D2_set_C_seed42"], "표본 A·B·C (test F1)")
summary(["D2_set_A_seed42", "D2_set_A_seed1", "D2_set_A_seed2"], "시드 42·1·2 (test F1)")
summary(["D2_set_B_seed42", "D2_set_C_seed42"], "(서버 결과 합치기 전) 표본 B·C")
summary(["D2_set_A_seed1", "D2_set_A_seed2"], "(서버 결과 합치기 전) 시드 1·2")
results_table(list(rs.values()))[["exp", "dev_F1", "test_F1", "macro_F1", "F1_NW", "F1_TR", "F1_FD", "train_min"]]

## 4. 저장 확인 후 런타임 반납
결과 파일이 드라이브에 다 있는지 확인하고, 다 있으면 **런타임을 반납**해 Colab 사용량(컴퓨팅 단위)을 아낀다. 하나라도 없으면 반납하지 않고 멈춘다. "모두 실행"으로 돌려 두면 끝까지 간 뒤 자동으로 반납된다.

드라이브 쓰기는 비동기라 반납 직전에 `drive.flush_and_unmount()` 로 저장을 마무리한다. 체크포인트 폴더(내 드라이브/ner_ckpt)는 실험이 끝날 때 자동으로 지워진다.

In [ ]:
ner_lib.verify_saved_models()   # 이번에 다룬 실험의 최종 가중치가 하나라도 없으면 여기서 멈추고 반납하지 않음
print("가중치 위치:", MODELS)
missing = []
for exp in RUNS:
    for f in (f"{RESULTS}/{exp}.json", f"{RESULTS}/scores/{exp}_dev.json", f"{RESULTS}/scores/{exp}_test.json",
              f"{RESULTS}/preds/{exp}_dev.jsonl", f"{RESULTS}/preds/{exp}_test.jsonl",
              f"{RESULTS}/errors/{exp}_dev.jsonl", f"{RESULTS}/errors/{exp}_test.jsonl"):
        if not os.path.exists(f): missing.append(f.replace(RESULTS, "results"))
if missing:
    print("저장 안 된 파일이 있어 반납하지 않습니다:", missing[:10])
else:
    print("결과 파일 모두 저장됨 (실험", len(RUNS), "개). 드라이브 저장을 마무리하고 런타임을 반납합니다.")
    drive.flush_and_unmount()
    from google.colab import runtime
    runtime.unassign()